In [3]:
import numpy as np
from matplotlib import pyplot as plt

In [5]:
users = []
movies = []
ratings = []
with open("../week1/movieLense-100k/ratings.csv") as f:
    for line in f:
        temp = (line.strip().split(","))
        users.append(temp[0])
        movies.append(temp[1])
        ratings.append(temp[2])

users.pop(0)
movies.pop(0)
ratings.pop(0)
print(len(users))
print(len(movies))
print(len(ratings))

print(f"{len(users)} x {len(movies)}")

100836
100836
100836
100836 x 100836


In [ ]:
def softImpute(Matrix, observation, sigma, mat_iter=50):

    Matrix = Matrix.copy().astype(float)

    # average rating in the TRAINING observations only
    meanRating = np.mean(Matrix[observation])

    # centre the observed ratings around 0
    centeredMatrix = Matrix.copy()
    centeredMatrix[observation] -= meanRating

    # current estimate of the centred matrix
    estimate = np.zeros_like(Matrix)

    for _ in range(mat_iter):

        # real centred values where observed, current guess everywhere else
        filled = np.where(
            observation,
            centeredMatrix,
            estimate
        )

        # SVD
        U, S, VT = np.linalg.svd(
            filled,
            full_matrices=False
        )

        # shrink singular values
        S = np.maximum(S - sigma, 0)

        # rebuild centred matrix
        new_estimate = U @ np.diag(S) @ VT

        # check convergence
        if np.linalg.norm(new_estimate - estimate) < 0.001:
            estimate = new_estimate
            break

        estimate = new_estimate

    # add the average rating back
    prediction = estimate + meanRating

    return prediction

In [32]:
def fitSigma(Matrix, trainObservation, validationObservation, mat_iter=50):

    possibleSigma = [
        1e-6,
        1e-5,
        1e-4,
        1e-3,
        1e-2,
        1e-1,
        1,
        10,
        100,
        1000
    ]

    bestSigma = None
    bestError = float("inf")

    sigmaResults = []
    rmseResults = []

    for sigma in possibleSigma:

        prediction = softImpute(
            Matrix,
            trainObservation,
            sigma,
            mat_iter
        )

        prediction = np.clip(prediction, 1, 5)

        rows, cols = np.where(validationObservation)

        errors = []

        for row, col in zip(rows, cols):

            realRating = Matrix[row][col]
            predictedRating = prediction[row][col]

            errors.append(
                (realRating - predictedRating) ** 2
            )

        rmse = np.sqrt(np.mean(errors))

        print(
            f"sigma = {sigma:.0e}, "
            f"RMSE = {rmse:.4f}"
        )

        sigmaResults.append(sigma)
        rmseResults.append(rmse)

        if rmse < bestError:

            bestError = rmse
            bestSigma = sigma

    print()
    print("Best sigma:", bestSigma)
    print("Best validation RMSE:", bestError)

    return bestSigma, sigmaResults, rmseResults

In [33]:
def test(Matrix, observation, testObservation, sigma, mat_iter=50):

    prediction = softImpute(Matrix, observation, sigma, mat_iter)

    prediction = np.clip(prediction, 1, 5)

    rows, cols = np.where(testObservation)

    errors = []

    for row, col in zip(rows, cols):

        realRating = Matrix[row][col]
        predictedRating = prediction[row][col]

        errors.append((realRating - predictedRating) ** 2)

    rmse = np.sqrt(np.mean(errors))

    print("Test RMSE:", rmse)

    return rmse

In [34]:
uniqueUsers = sorted(set(users), key=int)
uniqueMovies = sorted(set(movies), key=int)

userIndex = {}
movieIndex = {}

for i, user in enumerate(uniqueUsers):
    userIndex[user] = i

for i, movie in enumerate(uniqueMovies):
    movieIndex[movie] = i


Matrix = np.zeros(
    (len(uniqueUsers), len(uniqueMovies))
)

observation = np.zeros(
    Matrix.shape,
    dtype=bool
)


for user, movie, rating in zip(users, movies, ratings):

    u = userIndex[user]
    m = movieIndex[movie]

    Matrix[u][m] = float(rating)

    observation[u][m] = True


print("Matrix shape:", Matrix.shape)
print("Observed ratings:", np.sum(observation))

Matrix shape: (610, 9724)
Observed ratings: 100836


In [37]:
rows, cols = np.where(observation)

numberOfRatings = len(rows)

indices = np.arange(numberOfRatings)

rng = np.random.default_rng(42)

rng.shuffle(indices)


trainEnd = int(numberOfRatings * 0.7)

validationEnd = int(numberOfRatings * 0.9)


trainIndices = indices[:trainEnd]

validationIndices = indices[trainEnd:validationEnd]

testIndices = indices[validationEnd:]


trainObservation = np.zeros(
    Matrix.shape,
    dtype=bool
)

validationObservation = np.zeros(
    Matrix.shape,
    dtype=bool
)

testObservation = np.zeros(
    Matrix.shape,
    dtype=bool
)


trainObservation[
    rows[trainIndices],
    cols[trainIndices]
] = True


validationObservation[
    rows[validationIndices],
    cols[validationIndices]
] = True


testObservation[
    rows[testIndices],
    cols[testIndices]
] = True


print("Train:", np.sum(trainObservation))
print("Validation:", np.sum(validationObservation))
print("Test:", np.sum(testObservation))

Train: 70585
Validation: 20167
Test: 10084


In [38]:
mat_iter = 50

bestSigma, sigmaValues, validationRMSE = fitSigma(Matrix, trainObservation, validationObservation, mat_iter)

finalObservation = (trainObservation | validationObservation)

testRMSE = test(Matrix, finalObservation, testObservation, bestSigma, mat_iter)

sigma = 1e-06, RMSE = 1.0438
sigma = 1e-05, RMSE = 1.0438
sigma = 1e-04, RMSE = 1.0438
sigma = 1e-03, RMSE = 1.0436
sigma = 1e-02, RMSE = 1.0421
sigma = 1e-01, RMSE = 1.0290
sigma = 1e+00, RMSE = 0.9746
sigma = 1e+01, RMSE = 0.9166
sigma = 1e+02, RMSE = 1.0438
sigma = 1e+03, RMSE = 1.0438

Best sigma: 10
Best validation RMSE: 0.9165827864314383
Test RMSE: 0.8911892366185813


In [20]:
def meanBaseline(Matrix, trainObservation, testObservation):

    # average of all observed training ratings
    meanRating = np.mean(Matrix[trainObservation])

    rows, cols = np.where(testObservation)

    errors = []

    for row, col in zip(rows, cols):

        realRating = Matrix[row][col]

        errors.append(
            (realRating - meanRating) ** 2
        )

    rmse = np.sqrt(np.mean(errors))

    print("Mean rating:", meanRating)
    print("Mean baseline RMSE:", rmse)

    return rmse

In [ ]:
meanBaseline(
    Matrix,
    finalObservation,
    testObservation
)

Mean rating: 3.4989862482369536
Mean baseline RMSE: 1.0340549579457643


np.float64(1.0340549579457643)